# Murphy 1 · Chapter 5 · Practice

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app/#M5). Each exercise has a question, a hint, and an answer that stays hidden until you click.

**How to use:** try the starter cell first (fill in the `...` TODOs), then compare with the hidden solution cell (double-click its title to see the code), then open the answer.

## 5.1 · ROC and AUC from scratch  
*Section §5.1 · code*

Simulate 1,000 negative scores ~ N(0,1) and 1,000 positive scores ~ N(1.5,1). Sweep thresholds to build the ROC curve, compute its area with the trapezoid rule, and compare with the fraction of (positive, negative) pairs where the positive scores higher.

<details><summary><b>Hint</b></summary>

For each threshold t: TPR = mean(pos ≥ t), FPR = mean(neg ≥ t). Sort by FPR before integrating.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(7)
neg = rng.normal(0, 1, 1000); pos = rng.normal(1.5, 1, 1000)
ts = np.linspace(-5, 7, 400)
tpr = ...  # TODO: hit rate at each threshold
fpr = ...  # TODO: false-alarm rate at each threshold
o = np.argsort(fpr)
auc = np.trapezoid(tpr[o], fpr[o]) if hasattr(np, "trapezoid") else np.trapz(tpr[o], fpr[o])
pairs = ...  # TODO: P(random positive > random negative)
print(f"AUC (trapezoid) {auc:.4f}   P(pos > neg) {pairs:.4f}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(7)
neg = rng.normal(0, 1, 1000); pos = rng.normal(1.5, 1, 1000)
ts = np.linspace(-5, 7, 400)
tpr = np.array([(pos >= t).mean() for t in ts])  # hit rate at each threshold
fpr = np.array([(neg >= t).mean() for t in ts])  # false-alarm rate at each threshold
o = np.argsort(fpr)
auc = np.trapezoid(tpr[o], fpr[o]) if hasattr(np, "trapezoid") else np.trapz(tpr[o], fpr[o])
pairs = (pos[:, None] > neg[None, :]).mean()  # P(random positive > random negative)
print(f"AUC (trapezoid) {auc:.4f}   P(pos > neg) {pairs:.4f}")

<details><summary><b>Answer and explanation</b></summary>

Both numbers come out ≈ 0.87 (theory Φ(1.5/√2) ≈ 0.856; the gap is sampling noise). The area under the ROC curve equals the probability that a random positive outranks a random negative, which is why it doesn't depend on any threshold.

Expected output:
```
AUC (trapezoid) 0.8695   P(pos > neg) 0.8698
```

</details>

## 5.2 · The cost-optimal threshold  
*Section §5.1 · derive*

A false negative costs c_FN and a false positive costs c_FP. Given p = P(y = 1 | x), when should you predict 1? Work out the threshold for c_FN = 9, c_FP = 1.

<details><summary><b>Hint</b></summary>

Expected cost of predicting 1 is (1 − p)·c_FP; of predicting 0 is p·c_FN.

</details>

✍️ *Write your answer here (double-click to edit).*

<details><summary><b>Worked answer</b></summary>

Predict 1 when (1 − p)c_FP < p·c_FN ⇔ p > c_FP / (c_FP + c_FN). With c_FN = 9 and c_FP = 1, the threshold is 1/10 = 0.1: if missing a positive is 9× worse, you flag anything with more than a 10% chance. 0.5 is only right when the costs are equal (Murphy §5.1.2).

</details>

## 5.3 · Cross-validation from scratch  
*Section §5.4 · code*

Using the noisy-sine data (30 points), implement 5-fold cross-validation by hand to pick the polynomial degree from 0–10. Print the CV error for each degree.

<details><summary><b>Hint</b></summary>

Shuffle indices once, split into 5 folds; for each fold train on the other 4.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(8)
x = rng.random(30); y = np.sin(2 * np.pi * x) + rng.normal(0, 0.2, 30)
folds = np.array_split(rng.permutation(30), 5)
for d in range(11):
    errs = []
    for k in range(5):
        te = folds[k]; tr = ...  # TODO: all other folds
        c = np.polyfit(x[tr], y[tr], d)
        errs.append(np.mean((np.polyval(c, x[te]) - y[te]) ** 2))
    print(f"degree {d:2d}  CV error {np.mean(errs):8.4f}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(8)
x = rng.random(30); y = np.sin(2 * np.pi * x) + rng.normal(0, 0.2, 30)
folds = np.array_split(rng.permutation(30), 5)
for d in range(11):
    errs = []
    for k in range(5):
        te = folds[k]; tr = np.concatenate([folds[j] for j in range(5) if j != k])  # all other folds
        c = np.polyfit(x[tr], y[tr], d)
        errs.append(np.mean((np.polyval(c, x[te]) - y[te]) ** 2))
    print(f"degree {d:2d}  CV error {np.mean(errs):8.4f}")

<details><summary><b>Answer and explanation</b></summary>

CV error is high for degree 0 (too simple), lowest at degree 3 on this data, bumpy in the middle (30 points make CV noisy) and explodes for degree 7+. Unlike training error, CV error is measured on held-out points, so it can choose a degree without touching a test set.

Expected output:
```
degree  0  CV error   0.4925
degree  1  CV error   0.1950
degree  2  CV error   0.2008
degree  3  CV error   0.1146
degree  4  CV error   0.2488
degree  5  CV error   0.1472
degree  6  CV error   0.3307
degree  7  CV error   1.5179
degree  8  CV error   0.4745
degree  9  CV error   0.8458
degree 10  CV error  20.4529
```

</details>

## 5.4 · What a p-value is not  
*Section §5.5 · think*

A study reports p = 0.03 for 'the drug works'. Which statements are true? (a) There's a 3% chance the drug doesn't work. (b) If the drug did nothing, results at least this extreme would happen about 3% of the time. (c) The effect is large. (d) There's a 97% chance the result will replicate.

<details><summary><b>Hint</b></summary>

A p-value is computed assuming the null hypothesis is true.

</details>

✍️ *Write your answer here (double-click to edit).*

<details><summary><b>Worked answer</b></summary>

Only (b). A p-value is P(data at least this extreme | no effect). It isn't P(no effect | data); that would need a prior, i.e. Bayes' rule. It says nothing about effect size, and doesn't give a replication probability (Murphy §5.5.4).

</details>